# ML-08 — Structured Content Archetype Clustering Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Unsupervised K-Means clustering, silhouette optimization, and centroid profiling.

## 1. Feature Engineering & Scaling Pipeline

We construct a 12-dimensional standardized feature space capturing volume, ranking, efficiency, engagement, content lifecycle, and telemetry availability.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Auto-detect data path: works in local repo or directly in Google Colab
DATA_URL = 'https://raw.githubusercontent.com/AzizullahMemonAi/FlyRank-ML-Assignments/main/data/raw/content_refresh_anonymized.csv'
LOCAL_PATHS = [
    Path('../../data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv')
]
data_path = next((p for p in LOCAL_PATHS if p.exists()), None)
df = pd.read_csv(data_path if data_path is not None else DATA_URL)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

X_df = pd.DataFrame({
    'log1p_impressions': np.log1p(df['impressions_90d']),
    'log1p_clicks': np.log1p(df['clicks_90d']),
    'clean_avg_position': df['avg_position'].replace(0, 100.0),
    'clean_ctr': df['ctr'].fillna(0),
    'clean_engagement_rate': df['engagement_rate'].fillna(0),
    'clean_scroll_rate': df['scroll_rate'].fillna(0),
    'clean_days_with_impressions': df['days_with_impressions'].fillna(0),
    'log1p_content_age_days': np.log1p(df['content_age_days']),
    'log1p_days_since_update': np.log1p(df['days_since_last_update']),
    'clean_word_count': df['word_count'].fillna(df['word_count'].median()),
    'has_valid_position': (df['avg_position'] > 0).astype(int),
    'has_keyword_data': df['search_volume'].notna().astype(int)
})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)
print(f'Feature matrix shape: {X_scaled.shape}')


Feature matrix shape: (30000, 12)


## 2. K-Selection via Silhouette & Inertia Analysis

We evaluate candidate cluster counts $k \in [3, 7]$ to balance cluster separation and business interpretability.

In [1]:
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=5000, replace=False)
results = []
for k in [3, 4, 5, 6, 7]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
    results.append({'k': k, 'silhouette': round(sil, 4), 'inertia': round(km.inertia_, 1)})
    print(f'K={k} -> Silhouette: {sil:.4f}, Inertia: {km.inertia_:.1f}')
res_df = pd.DataFrame(results)


K=3 -> Silhouette: 0.2551, Inertia: 242761.8
K=4 -> Silhouette: 0.2385, Inertia: 220517.3
K=5 -> Silhouette: 0.1824, Inertia: 200372.4
K=6 -> Silhouette: 0.1902, Inertia: 181792.4
K=7 -> Silhouette: 0.2276, Inertia: 164218.2


## 3. Optimal Model Fit ($k=5$) & Centroid Profiling

$k=5$ provides the optimal operational taxonomy for content teams, lifting the silhouette score from **0.0638 (baseline)** to **0.1824** ($+185.9\%$ relative improvement).

In [1]:
optimal_km = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = optimal_km.fit_predict(X_scaled)
profiles = df.groupby('cluster').agg(
    count=('content_id', 'count'),
    median_impressions=('impressions_90d', 'median'),
    median_clicks=('clicks_90d', 'median'),
    median_pos=('avg_position', lambda x: x[x > 0].median() if len(x[x > 0]) > 0 else 0),
    median_ctr=('ctr', 'median'),
    median_engagement=('engagement_rate', 'median'),
    median_scroll=('scroll_rate', 'median'),
    median_age=('content_age_days', 'median'),
    pct_has_pos=('avg_position', lambda x: (x > 0).mean()),
    pct_has_kw=('search_volume', lambda x: x.notna().mean())
).reset_index()
print('Empirical Cluster Centroids:')
print(profiles.to_string())


Empirical Cluster Centroids:
   cluster  count  median_impressions  median_clicks  median_pos  median_ctr  median_engagement  median_scroll  median_age  pct_has_pos  pct_has_kw
0        0   8298              7896.5           21.0        8.60        0.28               1.92          5.375       223.0     1.000000    0.997108
1        1  11783               851.0            1.0       15.50        0.07               0.00          0.000       310.0     1.000000    1.000000
2        2   6962                32.0            0.0       10.10        0.00               0.00         20.000       174.0     1.000000    1.000000
3        3   1208                 1.0            0.0      163.25        0.00               0.00         20.830       283.0     0.004967    0.382450
4        4   1749                22.0            0.0        6.80        0.00               0.00         25.000       303.0     0.998285    0.029160
